# LLM is a NN trained to predict the next word.

The core task:

```text
Input: "The capital of France is"
            ↓
           LLM
            ↓
OUTPUT: ("Paris" 0.82, "a" 0.05, ...)
            ↓
Uses softmax to get the next word with the highest probability
```

# "Large"?

Number of parameters (**weights + biases**)

```text
MNIST      60,000

VGG16  | 138 million   | 2014 | 1.2M images
GPT-2 | 1.5 billion    | 2019 | 40GB text
GPT-3 | 175 billion    | 2020 | 570GB text
GPT-4 | ~1.7 trillion  | 2023 | Trillion of tokens
GPT-5 | Even larger    | 2025 | even more data
```

## 2017–2023

1. **The Transformer Architecture (2017)**  
   *"Attention Is All You Need" paper*
   - Enabled parallel training (**much faster!**)
   - Better at capturing long-range dependencies

2. **Massive Compute**
   - GPUs/TPUs became powerful enough
   - Companies invested billions in training
   - GPT-3 training: ~$4.6 million in compute

3. **Massive Data**
   - The internet has petabytes of text
   - Books, Wikipedia, code, conversations
   - Models learned from human knowledge

---

## RNNs (1980s) / LSTMs (1997)

### RNN

```text
"The" -> "cat" -> "sat" -> "on" -> "the" -> ???

[RNN] -> [RNN] -> [RNN] -> [RNN] -> [RNN] -> predict
state -> state -> state -> state -> state
```

---

## Transformer

```text
"The" <-> "cat" <-> "sat" <-> "on" <-> "the"
                    ALL AT ONCE

[SELF-ATTENTION]

Every word looks at every word!

                ↓
             Predict
```

**Transformer = Neural Network that uses ATTENTION instead of recurrence.**

It transforms input sequences into output sequences using attention to understand relationships between all positions (words).

### Key Advantages

1. **Parallel**
2. **Attention:** every word can attend to every other word
3. **Scalable**
4. **Versatile**

In [16]:
import torch                         # PyTorch - the deep learning framework
import torch.nn.functional as F      # F contains functions like softmax, relu, etc.
import numpy as np
import matplotlib.pyplot as plt

# Hugging Face transformers - library for working with pre-trained LLMs
from transformers import AutoTokenizer, AutoModelForCausalLM


import warnings
warnings.filterwarnings("ignore")

In [17]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7030.25it/s]


In [18]:
prompt = "The capital of France is"

# Step 1: convert the text to token IDs (numbers)

inputs = tokenizer(prompt, return_tensors="pt") # pt means return pytorch tensor, tf, np
print(f"Input IDs shape: {inputs['input_ids'].shape}") # batch size, seq len
print(f"Input IDs: {inputs['input_ids']}")

Input IDs shape: torch.Size([1, 5])
Input IDs: tensor([[ 464, 3139,  286, 4881,  318]])


In [19]:
# Step 2: Pass through the model
with torch.no_grad(): # no_grad disable gradient computation, saves memory
  outputs = model(**inputs) # **inputs unpacks dictionary to keyword args

# Step 3: Get prediticions for the LAST position (next word)
# outputs.logits # [batch_size, sequence_length, vocab_size]

# We want : [0, -1, :]

logits = outputs.logits[0, -1, :]
print(f"Logits shape: {logits.shape}")

Logits shape: torch.Size([50257])


In [20]:
# logits # raw scores

# Step 4: Convert logits to probabilities
probabilities = F.softmax(logits, dim=-1) # for 2D [batch, vocab], dim=01 would normalize each row indpenedently

# Step 5: Get the top 5 predictions

top_probabilities, top_indices = torch.topk(probabilities, k=5)

print(f"PROMPT: {prompt}")

for probability, index in zip(top_probabilities, top_indices):
  word = tokenizer.decode(index)
  print(f"'{word}' -> {probability:.1%}")

PROMPT: The capital of France is
' the' -> 8.5%
' now' -> 4.8%
' a' -> 4.6%
' France' -> 3.2%
' Paris' -> 3.2%


NNs understand only  numbers so we need to convert words into numbers. This is done using **embeddings** and **tokenization**.
``` Example of tokenization and embeddings ```
1. **Tokenization**: Splitting text into smaller units (tokens) like words or subwords.
   - Example: "The capital of France is Paris." → ["The", "capital", "of", "France", "is", "Paris", "."]
2. **Embeddings**: Mapping tokens to high-dimensional vectors that capture semantic meaning.




# Tokenization: Converting Text to Numbers

Neural Networks understand only numbers. We need a way to convert text into numbers and vice versa.

**Example Input:**
`"Hello, how are you?"`

---

**Step 1: Split words into pieces (tokens)**
- `"Hello"`, `","`, `" how"`, `" are"`, `" you"`, `"?"`

**Step 2: Look up each piece in the vocabulary**
| Token | Token ID |
|-------|----------|
| `"Hello"` | `15496` |
| `","` | `11` |
| `" how"` | `703` |
| `" are"` | `...` |
| `" you"` | `...` |
| `"?"` | `...` |

**Step 3: Resulting numerical sequence (ready for the NN)**
```python
[15496, 11, 703, ...]

## Tokenization Levels in NLP

---

### 1. **WORD-LEVEL**
`"I love Machine Learning"` $\rightarrow$ `["I", "love", "Machine", "Learning"]`

> ⚠️ **Problems:**
* **Huge Vocabulary:** Scales to millions of unique words, consuming massive memory.
* **Out-of-Vocabulary (OOV):** Unknown words completely fail to process.
* **Mispellings:** Highly fragile (e.g., `"learninggg"` $\rightarrow$ `???`).

---

### 2. **Character-level**
`"Hello"` $\rightarrow$ `["H", "e", "l", "l", "o"]`

> ⚠️ **Problems:**
* **Long Sequences:** A brief 10-word sentence explodes into $50+$ characters, taxing the model's context window.
* **Loss of Meaning:** Harder for the model to capture semantic relationships (e.g., splitting `"cat"` into `c-a-t` strips its immediate conceptual meaning).
* **Performance:** Significantly slower to process during training and inference.

---

### 3. **SUBWORD-level** *(Used by Modern LLMs)*
Strikes a perfect balance by breaking rare words into common building blocks while keeping frequent words intact.

* `"unbelievable"` $\rightarrow$ `["un", "believ", "able"]`
* `"tokenziation"` $\rightarrow$ `["token", "ization"]`
* `"Hello"` $\rightarrow$ `["Hello"]`

Best of both worlds:
1. * **Vocabulary Size:** Smaller than word-level, reducing memory footprint and stays manageable.
2. * **OOV Handling:** Can represent unseen words by combining known subwords.
3. common subwords like `"ing"`, `"tion"`, `"un"` are shared across many words, allowing the model to generalize better. And common words stay intact, preserving their meaning.
4. Rare words are broken down into subwords, allowing the model to understand and generate them even if they were not seen during training.




In [21]:
text = "Hello, how are you?"

token_ids = tokenizer.encode(text)

print(f"Token IDs: {token_ids}")

Token IDs: [15496, 11, 703, 389, 345, 30]


In [22]:
tokenizer.convert_ids_to_tokens(token_ids)

['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']

In [23]:
examples = ["the", "hello", "transformer", "tokenization", "Pneumonia"]

for word in examples:
  tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(word))
  print(f"'{word}' -> {tokens}, {tokenizer.encode(word)}")

'the' -> ['the'], [1169]
'hello' -> ['hello'], [31373]
'transformer' -> ['trans', 'former'], [7645, 16354]
'tokenization' -> ['token', 'ization'], [30001, 1634]
'Pneumonia' -> ['P', 'neum', 'onia'], [47, 25668, 11339]


### Embeddings: Each Token Becomes a Vector

Embeddings convert each token into a vector (**list of numbers**) where similar words have similar vectors.

```text
            "cat" (9246)

                |
                v

           "cat"  →  [0.2, -0.5, 0.8, ...]
           "dog"  →  [0.3, -0.4, 0.7, ...]
```

The model learns these vectors during training.

**Similar words end up with similar embedding vectors.**